# Chapter 3 &mdash; `lstar`: the Computable Stand-in for Star

**Concept 3 of the Chapter 3 decomposition:** *Star Bounded at $n$, and `lstar`*

$L^*_n = L^n \cup L^*_{n-1}$ with base $\{\varepsilon\}$. The true $L^*$ <b>cannot</b> be computed, and the API refuses to pretend otherwise.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter3/Concept-Lstar-Bounded/Concept-Lstar-Bounded.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Since the real star is an infinite union, Jove works with **star up to $n$**:

$$L^*_n = L^n \cup L^*_{n-1}, \qquad L^*_0 = \{\varepsilon\}$$

```python
def lstar(L,n):
    return lunit() if n == 0 else lunion(lexp(L,n), lstar(L,n-1))
```

**$L^* = L^*_\infty$ cannot be computed.** Note there is deliberately **no one-argument
`lstar(L)`** &mdash; the bound is not optional.

## 2. Definitions

### The recursion, by hand

In [ ]:
def lstar_by_hand(L, n):
    return lunit() if n == 0 else lunion(lexp(L,n), lstar_by_hand(L,n-1))

### The bound is on the **exponent**, not on string length

In [ ]:
long_L = {'abcdef'}
print("lstar(long_L, 2) :", sorted(lstar(long_L, 2), key=len))
print("exponent bound 2, yet the longest string is", max(len(s) for s in lstar(long_L,2)), "symbols")

## 3. Tests

Hand version and Jove agree everywhere.

In [ ]:
ok = all(lstar_by_hand(L,n) == lstar(L,n)
         for L in [{'ab','bc'}, {'0'}, lphi(), lunit(), {'a','ab'}]
         for n in range(5))
print("lstar_by_hand == lstar :", ok)
assert ok

The book's worked example.

In [ ]:
print("lstar({'ab','bc'},2) :", sorted(lstar({'ab','bc'}, 2)))
assert lstar({'ab','bc'},2) == {'', 'ab', 'bc', 'abab', 'abbc', 'bcab', 'bcbc'}

Growing the bound never terminates the process. There is no `lstar(L)`.

In [ ]:
L = {'0','1'}
for n in range(6):
    print("bound %d -> %3d strings" % (n, len(lstar(L,n))))
print()
try:
    lstar(L)
except TypeError as e:
    print("lstar(L) with one argument ->", type(e).__name__)
    print("  Deliberate: the true star is not computable, so the bound is required.")

## 4. Exercises


1. `lstar(L,n)` calls `lexp(L,n)` then recurses. How many `lcat` calls in total?
2. Predict `lstar(lphi(), 5)` and `lstar(lunit(), 5)` before running them.
3. Why is "bound on the exponent" different from "bound on string length"? Give a
   language where they differ wildly.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter3/Concept-Lstar-Bounded')